# Semana 08: Deploy Automatizado na AWS EC2, Validação e Rollback Automático

## Módulo de Implantação em Nuvem e Resiliência — Fábrica Virtual Smart N1

Este notebook apresenta a automação da implantação de containers Docker em instâncias **AWS EC2**, a validação pós-deploy por meio de **Healthchecks HTTP** e o mecanismo de **Rollback Automático** em caso de falha na aplicação em produção.

### Objetivos de aprendizagem
- Compreender a arquitetura de deploy em instâncias virtuais AWS EC2 executando Docker Engine.
- Configurar automação SSH segura no GitHub Actions (`appleboy/ssh-action`) com chaves PEM criptografadas.
- Implementar verificação de saúde pós-deploy (*Healthcheck Probe*).
- Projetar e automatizar a estratégia de **Rollback Automático** para restaurar a versão anterior em caso de erro HTTP 5xx.
- Executar um simulador em Python do ciclo de deploy, healthcheck e rollback automatizado na AWS.

---


## 1. Fundamentação Teórica

### 1.1 O Ciclo de Deploy com Healthcheck e Rollback Automático

Em um ambiente de produção resiliente, o pipeline de CD não pode considerar o deploy concluído no momento em que o container inicia. É necessário realizar uma **sonda de validação de saúde (Healthcheck Probe)**:

```text
  +-----------------------------------------------------------------------------------------+
  |                PIPELINE DE DEPLOY AWS EC2 COM AUTO-ROLLBACK                             |
  |                                                                                         |
  |  1. GitHub Actions -> Conecta via SSH na EC2 da AWS                                     |
  |  2. Salva ID da versão atual em execução (v1.0.0)                                       |
  |  3. Executa `docker pull usuario/app:v1.1.0` & `docker run -d --name app_v2`           |
  |  4. Executa Healthcheck: curl -f http://ec2-ip:8080/health                              |
  |                                                                                         |
  |                                  /                     \                                |
  |                            (HTTP 200 OK)           (HTTP 500 / Timeout)                   |
  |                                 /                       \                               |
  |                                v                         v                              |
  |                    [Aprova Deploy & Remove v1]   [ROLLBACK AUTOMÁTICO REVOLVE!]         |
  |                                                  - Para container v2                    |
  |                                                  - Reinicia container v1 (v1.0.0)        |
  |                                                  - Alerta equipe via webhook            |
  +-----------------------------------------------------------------------------------------+
```

---

### 1.2 Script de Deployment e Healthcheck no Runner (`deploy_ec2.sh`)

```bash
#!/bin/bash
set -e

EC2_IP="${1}"
NOVA_TAG="${2}"
VERSAO_ANTERIOR=$(docker ps --filter "name=smartn1_app" --format "{{.Image}}")

echo "=== INICIANDO DEPLOY NA AWS EC2 (${EC2_IP}) ==="
docker pull smartn1oficial/telemetria-api:${NOVA_TAG}

# Parar container antigo e subir o novo
docker stop smartn1_app || true
docker rm smartn1_app || true
docker run -d --name smartn1_app -p 80:8000 smartn1oficial/telemetria-api:${NOVA_TAG}

echo "=== EXECUTANDO HEALTHCHECK PROBE ==="
sleep 5
HTTP_STATUS=$(curl -s -o /dev/null -w "%{http_code}" http://localhost/health || echo "500")

if [ "$HTTP_STATUS" -eq 200 ]; then
  echo "✅ Deploy concluído com sucesso! Healthcheck HTTP 200 OK."
else
  echo "❌ FALHA NO HEALTHCHECK (Status: ${HTTP_STATUS})! INICIANDO ROLLBACK..."
  docker stop smartn1_app || true
  docker rm smartn1_app || true
  docker run -d --name smartn1_app -p 80:8000 ${VERSAO_ANTERIOR}
  echo "⚠️ Rollback concluído! Versão anterior (${VERSAO_ANTERIOR}) restaurada."
  exit 1
fi
```

---


## 2. Prática — Simulador de Deploy AWS EC2 e Motor de Rollback em Python

Nesta prática, executaremos uma simulação de deploy na AWS EC2 onde a nova versão enviada apresenta um erro intermitente de inicialização, disparando o processo de validação e rollback automático.

In [ ]:
import time
import random

def simular_deploy_aws_ec2(imagem_atual, imagem_nova, simular_falha_healthcheck=False):
    print(f"[AWS EC2] Conectado na instância ec2-54-12-34-56.compute.amazonaws.com")
    print(f"[AWS EC2] Versão atual em execução: '{imagem_atual}'")
    print(f"[AWS EC2] Baixando nova versão: '{imagem_nova}'...")
    
    # Simular parada da versão atual e subida da nova
    print(f"[AWS EC2] Parando container '{imagem_atual}' e iniciando '{imagem_nova}'...")
    time.sleep(0.5)
    
    # Simular Healthcheck HTTP
    print(f"[HEALTHCHECK] Testando GET http://localhost/health...")
    if simular_falha_healthcheck:
        status_http = 500
        msg_erro = "Internal Server Error - Falha na conexão com banco de dados"
    else:
        status_http = 200
        msg_erro = "OK"
        
    if status_http == 200:
        print(f"[HEALTHCHECK SUCCESS] HTTP 200 OK! Deploy concluído com sucesso.")
        return True, imagem_nova
    else:
        print(f"[HEALTHCHECK FAILED] HTTP {status_http} ({msg_erro})!")
        print(f"[AUTO-ROLLBACK] Restaurando versão estável anterior '{imagem_atual}'...")
        time.sleep(0.5)
        print(f"[AUTO-ROLLBACK SUCCESS] Container '{imagem_atual}' em execução novamente na porta 80!")
        return False, imagem_atual

# Execução do teste de deploy com simulação de falha e rollback
sucesso, versao_final = simular_deploy_aws_ec2(
    imagem_atual="smartn1/telemetria-api:v1.0.0",
    imagem_nova="smartn1/telemetria-api:v1.1.0-buggy",
    simular_falha_healthcheck=True
)

print(f"\n=== STATUS FINAL DA INSTÂNCIA AWS EC2 ===")
print(f"- Sucesso do Deploy: {sucesso}")
print(f"- Versão Rodando em Produção: {versao_final}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Por que o deploy automatizado via SSH em uma instância AWS EC2 deve obrigatoriamente incluir uma etapa de **Healthcheck** (verificação de resposta HTTP) antes de considerar a pipeline concluída?

### Questão 2
Explique a mecânica de um **Rollback Automático**. Quais comandos Docker devem ser executados na instância EC2 quando o teste de saúde da nova versão falha?

### Questão 3
Como armazenar a chave privada de acesso SSH (`.pem`) da instância AWS EC2 no **GitHub Actions Secrets** de forma segura, garantindo que ela não seja exposta nos logs de execução do pipeline?
